# Retrieval Augmented Generation (RAG) mit Ollama & Chroma

* was RAG ist und **was nicht**
* wie Dokumente indexiert werden
* wie Retrieval funktioniert (Top-K Chunks)
* warum das LLM **nur mit Kontext** antworten kann
* wie man RAG systematisch debuggt

### Architekturüberblick 

**RAG-Pipeline**

1. Dokumente einlesen (PDF / Text)
2. Text in Chunks aufteilen
3. Chunks embedden
4. Embeddings speichern (Vector Store)
5. Frage embedden
6. ähnlichste Chunks finden (Top-K)
7. Prompt mit Kontext bauen
8. Antwort vom LLM erzeugen

👉 Wichtig:
**Das Modell weiss nichts über die Dokumente.**
Alles Wissen kommt ausschliesslich aus dem Kontext.


### Libraries und Konfiguration 

* `CHUNK_SIZE` und `TOP_K` sind **keine Details**, sondern steuern die Qualität direkt.
* RAG ist **Daten-Engineering**, nicht Prompt-Magie.


In [ ]:
import os

OLLAMA_BASE_URL = "http://localhost:11434"
CHAT_MODEL = "llama3.1:8b-instruct-q4_K_M"
EMBED_MODEL = "nomic-embed-text"

PERSIST_DIR = "./chroma_db"
COLLECTION_NAME = "docs"

CHUNK_SIZE = 900
CHUNK_OVERLAP = 150
TOP_K = 6


---

### Textaufbereitung & Chunking (reine Vorbereitung)

Bevor wir irgendetwas embedden:

* Sonderzeichen entfernen
* Whitespace normalisieren
* Text in **überlappende Chunks** aufteilen

Warum Overlap?
➡️ Damit Sätze nicht „abgeschnitten“ werden.

In [ ]:
import re
import hashlib
from typing import List, Dict, Iterable
from pypdf import PdfReader

def sha1(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest()

def clean_text(s: str) -> str:
    s = s.replace("\x00", " ")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def chunk_text(text: str) -> List[str]:
    text = clean_text(text)
    chunks, start = [], 0
    while start < len(text):
        end = min(len(text), start + CHUNK_SIZE)
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = end - CHUNK_OVERLAP
    return chunks

def read_pdf(path: str) -> str:
    reader = PdfReader(path)
    return "\n".join(p.extract_text() or "" for p in reader.pages)

def read_text(path: str) -> str:
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            with open(path, encoding=enc, errors="ignore") as f:
                return f.read()
        except Exception:
            pass
    return ""


---

### Ollama API 

Wir verwenden **zwei unterschiedliche Endpunkte**:

* `/api/embeddings` → Vektoren
* `/v1/chat/completions` → Textgenerierung

Diese Trennung ist **zentral für RAG**.

In [ ]:
import requests

def ollama_embed(texts: List[str]) -> List[List[float]]:
    vectors = []
    for t in texts:
        r = requests.post(
            f"{OLLAMA_BASE_URL}/api/embeddings",
            json={"model": EMBED_MODEL, "prompt": t},
            timeout=120
        )
        r.raise_for_status()
        vectors.append(r.json()["embedding"])
    return vectors

def ollama_chat(prompt: str) -> str:
    r = requests.post(
        f"{OLLAMA_BASE_URL}/v1/chat/completions",
        json={
            "model": CHAT_MODEL,
            "temperature": 0.2,
            "messages": [
                {"role": "system", "content": "Antworte präzise. Nutze nur den Kontext."},
                {"role": "user", "content": prompt},
            ],
        },
        timeout=300,
    )
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]


---

### Chroma Storage

Wir verwenden **ChromaDB** als lokalen Vector Store:

* speichert Embeddings
* erlaubt Similarity Search
* persistent auf Disk

In [ ]:
import chromadb
from chromadb.config import Settings

def get_collection():
    client = chromadb.PersistentClient(
        path=PERSIST_DIR,
        settings=Settings(anonymized_telemetry=False)
    )
    return client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"}
    )


---

### RAG Core – Ingest 

**Ingest bedeutet:**

* Dokument lesen
* in Chunks zerlegen
* Chunks embedden
* Embeddings speichern

➡️ Das LLM ist hier **nicht beteiligt**.

In [ ]:
def ingest(paths: List[str]):
    col = get_collection()
    ids, texts, metas = [], [], []

    for p in paths:
        if p.lower().endswith(".pdf"):
            content = read_pdf(p)
        else:
            content = read_text(p)

        for i, chunk in enumerate(chunk_text(content)):
            ids.append(sha1(f"{p}:{i}:{chunk[:80]}"))
            texts.append(chunk)
            metas.append({"source": p, "chunk": str(i)})

    print(f"Chunks: {len(texts)}")
    embs = ollama_embed(texts)

    try:
        col.delete(ids=ids)
    except Exception:
        pass

    col.add(
        documents=texts,
        embeddings=embs,
        metadatas=metas,
        ids=ids
    )

    print("Indexierung abgeschlossen.")



---

### Retrieval & Debugging

Jetzt kommt der **entscheidende Schritt**:

* 👉 **Wir schauen uns zuerst die Chunks an**,
* 👉 **erst danach** fragen wir das Modell.

In [ ]:
   
def retrieve(question: str, k: int = TOP_K):
    col = get_collection()
    q_emb = ollama_embed([question])[0]

    res = col.query(
        query_embeddings=[q_emb],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )

    return (
        res["documents"][0],
        res["metadatas"][0],
        res["distances"][0],
    )


Top-K Visualisierung 

In [ ]:
def show_top_k(docs, metas, dists, max_chars=500):
    for i, (d, m, dist) in enumerate(zip(docs, metas, dists), start=1):
        print(f"\n=== Treffer {i} ===")
        print(f"Quelle : {m['source']}")
        print(f"Chunk  : {m['chunk']}")
        print(f"Distanz: {dist:.4f}")
        print("-" * 80)
        print(d[:max_chars])


---

### Prompt-Building

Das LLM bekommt **keine Dokumente**, sondern **nur diesen Prompt**.

Wenn die Antwort falsch ist:
* ➡️ Prompt anschauen
* ➡️ Chunks anschauen
* ➡️ nicht das Modell beschuldigen


In [ ]:
def make_prompt(question, docs, metas):
    blocks = []
    for d, m in zip(docs, metas):
        blocks.append(
            f"[Quelle: {m['source']} | Chunk: {m['chunk']}]\n{d}"
        )

    return (
        "Beantworte die Frage nur anhand des Kontexts.\n\n"
        "KONTEXT:\n" + "\n\n---\n\n".join(blocks) +
        f"\n\nFRAGE:\n{question}\n\nANTWORT:"
    )

---

## Notebook-Use-Case

1. Dokument indexieren
2. Top-K Chunks prüfen
3. Antwort generieren

In [ ]:
# 1. Index bauen
ingest(["engineering-the-quantum-future.pdf"])


In [ ]:
# 2. Frage stellen + Chunks ansehen
docs, metas, dists = retrieve("Explain Quantum Computing", k=5)
show_top_k(docs, metas, dists)


In [ ]:
# 3. Antwort generieren
prompt = make_prompt("Explain Quantum Computing", docs, metas)
print(ollama_chat(prompt))
